# Previous Config

In [ ]:
!pip install -U ranx sentence-transformers datasets accelerate
!pip uninstall -y wandb

import argparse
import itertools
import json
import os
import re
import time
import tempfile
import unicodedata
import shutil
import logging
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from ranx import Qrels, Run, evaluate
from sentence_transformers import SentenceTransformer, losses, SentenceTransformerTrainer, SentenceTransformerTrainingArguments, InputExample, losses, util, evaluation
from sentence_transformers.training_args import BatchSamplers
from datasets import Dataset
from transformers import EarlyStoppingCallback


In [ ]:
# Configuración básica
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)
today = time.strftime("%Y-%m-%d")

In [ ]:
def set_directories():
  project_dir = Path(__name__).resolve().parents[1]
  project_dir = project_dir / 'content' / 'TalentCLEF-TaskA'
  date_dir = project_dir.parent / 'output' / today
  date_dir.mkdir(parents=True, exist_ok=True)

  # Crear subdirectorio incremental por ejecución (001, 002, ...)
  exec_dirs = sorted([d for d in date_dir.iterdir() if d.is_dir() and d.name.isdigit() and len(d.name) == 3])
  if exec_dirs and not any(exec_dirs[-1].iterdir()):
      output_dir = exec_dirs[-1]
  else:
      next_id = int(exec_dirs[-1].name) + 1 if exec_dirs else 1
      output_dir = date_dir / f"{next_id:03d}"
      output_dir.mkdir(exist_ok=True)

  return project_dir, output_dir

project_dir, output_dir = set_directories()

In [ ]:
!git clone https://github.com/davidgc14/TalentCLEF-TaskA.git
!cp ./TalentCLEF-TaskA/src/output/ranking_spanish_validation.csv {output_dir.parent.parent}
!rm -r sample_data/

fatal: destination path 'TalentCLEF-TaskA' already exists and is not an empty directory.
rm: cannot remove 'sample_data/': No such file or directory


## Evaluation functions

In [ ]:
def load_qrels(qrels_path):
    """
    Loads the qrels file (TREC format: q_id, iter, doc_id, rel)
    and converts it to a Qrels object.
    """
    qrels_df = pd.read_csv(qrels_path, sep="\t", header=None,
                           names=["q_id", "iter", "doc_id", "rel"],
                           dtype={"q_id": str, "doc_id": str, "rel":int})

    return Qrels.from_df(qrels_df, q_id_col="q_id", doc_id_col="doc_id", score_col="rel")

def load_run(run_path):
    """
    Loads the run file (TREC format: q_id, Q0, doc_id, rank, score, [tag])
    and converts it to a Run object.
    """
    run_df = pd.read_csv(run_path, sep=r"\s+", header=None)

    # Assign column names based on the number of columns
    if run_df.shape[1] == 5:
        run_df.columns = ["q_id", "Q0", "doc_id", "rank", "score"]
    elif run_df.shape[1] >= 6:
        run_df.columns = ["q_id", "Q0", "doc_id", "rank", "score", "tag"]
    else:
        raise ValueError("The run file does not have the expected format.")

    run_df["q_id"] = run_df.q_id.astype(str)
    run_df["doc_id"] = run_df.doc_id.astype(str)
    return Run.from_df(run_df, q_id_col="q_id", doc_id_col="doc_id", score_col="score")

def evaluate_run(qrels_path, run_path):
    """Evalúa un run y devuelve los resultados como dict."""
    qrels = load_qrels(qrels_path)
    run = load_run(run_path)
    metrics = ["map", "mrr", "ndcg", "precision@5", "precision@10", "precision@100"]
    return evaluate(qrels, run, metrics)

In [ ]:

# ========================
# DATA LOADING AND ENCODING
# ========================

def load_spanish_data(data_dir):
    """Load queries and corpus elements from Spanish data directory."""
    queries_path = data_dir / "queries"
    corpus_elements_path = data_dir / "corpus_elements"

    queries = pd.read_csv(queries_path, sep="\t")
    corpus_elements = pd.read_csv(corpus_elements_path, sep="\t")

    return (
        queries.q_id.to_list(),
        queries.jobtitle.to_list(),
        corpus_elements.c_id.to_list(),
        corpus_elements.jobtitle.to_list(),
    )


def encode_data(model, queries_texts, corpus_texts, model_name, device):
    """Encode queries and corpus using the model."""
    print('Encoding data:', model_name, 'on device:', device)
    query_embeddings = model.encode(queries_texts, convert_to_tensor=True, show_progress_bar=True, device=device)
    corpus_embeddings = model.encode(corpus_texts, convert_to_tensor=True, show_progress_bar=True, device=device)
    return query_embeddings, corpus_embeddings


def calculate_similarities_and_format_results(query_embeddings, corpus_embeddings, queries_ids, corpus_ids, model_name):
    """Calculate cosine similarities and format results in TREC format."""
    print('Calculating similarities and preparing results...')
    similarities = util.cos_sim(query_embeddings, corpus_embeddings).cpu().numpy()

    results = []
    for q_idx, q_id in enumerate(queries_ids):
        sorted_indices = np.argsort(-similarities[q_idx])  # Orden descendente
        for rank, c_idx in enumerate(sorted_indices):
            doc_id = corpus_ids[c_idx]
            score = similarities[q_idx, c_idx]
            results.append(f"{str(q_id)} Q0 {str(doc_id)} {rank+1} {score:.4f} {model_name}")
    return results


def run_evaluation_temp(qrels_path, results, model_name):
    """Run evaluation using temporary file without saving."""
    print('Evaluating Spanish monolingual performance...')

    # Crear archivo temporal
    with tempfile.NamedTemporaryFile(mode='w', suffix='.trec', delete=False, encoding='utf-8') as tmp_file:
        tmp_file.write("\n".join(results))
        tmp_path = tmp_file.name

    try:
        evaluation_results = evaluate_run(qrels_path, tmp_path)
    finally:
        # Eliminar archivo temporal
        os.unlink(tmp_path)

    return evaluation_results


def get_model_name(model):
    """Extract model name from the model object."""
    model_name = model[0].auto_model.config._name_or_path
    return model_name.split("/")[-1]


# ========================
# EVALUATION FUNCTION
# ========================

def spanish_monolingual_evaluation(model, device, source):
    """Evaluate model performance on Spanish monolingual data."""
    data_dir = project_dir / 'data' / source / 'spanish'
    qrels_path = data_dir / "qrels.tsv"

    print('Loading Spanish data...')
    queries_ids, queries_texts, corpus_ids, corpus_texts = load_spanish_data(data_dir)
    model_name = get_model_name(model)

    query_embeddings, corpus_embeddings = encode_data(model, queries_texts, corpus_texts, model_name, device)
    results = calculate_similarities_and_format_results(query_embeddings, corpus_embeddings, queries_ids, corpus_ids, model_name)

    evaluation_results = run_evaluation_temp(qrels_path, results, model_name)
    print('Spanish evaluation completed')
    return evaluation_results


# ========================
# SAVING RESULTS
# ========================

def save_spanish_results(evaluation_results, model_name, nickname, source):
    """Save Spanish monolingual evaluation results to JSON file."""
    print("Saving Spanish evaluation results...")

    json_path = output_dir / "results_spanish_monolingual.json"

    results_data = {
        "metadata": {
            "type": "spanish_monolingual",
            "model_name": model_name,
            "nickname": nickname,
            "source": source,
            "timestamp": today
        },
        "results": evaluation_results
    }

    with open(json_path, "w", encoding="utf-8") as jf:
        json.dump(results_data, jf, indent=2, ensure_ascii=False)

    print(f"Saved Spanish results to {json_path}")


# ========================
# RANKING
# ========================

def update_spanish_ranking(map_score, model_name, nickname, source):
    """Update Spanish-specific ranking CSV file with current execution results."""
    print("Updating Spanish ranking file...")

    ranking_file = output_dir.parent.parent / f"ranking_spanish_{source}.csv"
    execution_id = output_dir.name

    new_record = {
        'timestamp': today,
        'execution_id': execution_id,
        'model_name': model_name,
        'model_alias': nickname,
        'map_es_es': map_score
    }

    if ranking_file.exists():
        df = pd.read_csv(ranking_file)
    else:
        df = pd.DataFrame(columns=['timestamp', 'execution_id', 'model_name', 'model_alias', 'map_es_es'])

    # Double check for existing identical record
    comparison_cols = ['model_name', 'model_alias', 'map_es_es']

    if not df.empty:
        new_record_comparison = {k: new_record.get(k, np.nan) for k in comparison_cols}
        existing_records = df[comparison_cols].to_dict('records')

        for existing in existing_records:
            if all(abs(existing.get(k, np.nan) - new_record_comparison.get(k, np.nan)) < 1e-4
                   if isinstance(new_record_comparison.get(k), (float, int)) and not np.isnan(new_record_comparison.get(k, np.nan))
                   else existing.get(k) == new_record_comparison.get(k)
                   for k in comparison_cols):
                print("Ya existe un registro idéntico en el ranking español. No se agregará el nuevo registro.")
                return

    # Evitar warning de pandas con DataFrame vacío
    if df.empty:
        df = pd.DataFrame([new_record])
    else:
        df = pd.concat([df, pd.DataFrame([new_record])], ignore_index=True)

    df = df.sort_values('map_es_es', ascending=False).reset_index(drop=True)
    df.to_csv(ranking_file, index=False)

    print(f"Ranking español actualizado en {ranking_file}")


# ========================
# MAIN PIPELINE
# ========================

def run_spanish_evaluation(model_name, nickname, device, source):
    """Run complete Spanish monolingual evaluation pipeline."""
    model = SentenceTransformer(model_name, device=device)

    evaluation_results = spanish_monolingual_evaluation(model, device, source)
    save_spanish_results(evaluation_results, model_name, nickname, source)

    map_score = evaluation_results.get('map', np.nan)
    update_spanish_ranking(map_score, model_name, nickname, source)

    print(f"\n{'='*50}")
    print(f"Evaluación completada para {model_name}")
    print(f"MAP español-español: {map_score:.4f}")
    print(f"{'='*50}\n\n")

# Training path

## Config

In [ ]:
MODEL_NAME = 'paraphrase-multilingual-mpnet-base-v2'
BATCH_SIZE = 64
EPOCHS = 30
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# Rutas de salida para el entrenamiento Hugging Face
model_output_path = output_dir / 'finetuned_models' / MODEL_NAME
best_model_path = model_output_path / 'best_model'
logs_path = model_output_path / 'logs'

print(f"Directorio de proyecto: {project_dir}")
print(f"Guardando outputs en: {model_output_path}")

Directorio de proyecto: /content/TalentCLEF-TaskA
Guardando outputs en: /content/output/2025-11-26/001/finetuned_models/paraphrase-multilingual-mpnet-base-v2


## Data preparation

### Training data

In [ ]:
def normalize_text(text):
    if pd.isna(text): return ""
    text = str(text).lower().strip()
    text = unicodedata.normalize('NFC', text)
    # Reemplazar puntuación por espacio para no pegar palabras
    text = re.sub(r'[^\w\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

In [ ]:
training_file = project_dir / 'data' / 'training' / 'spanish' / 'taskA_training_es.tsv'

train_df = pd.read_csv(training_file, sep='\t', header=None)
train_df.columns = ['family_id', 'id', 'jobtitle_1', 'jobtitle_2']

In [ ]:
expanded = []
for _, row in train_df.iterrows():
    jt1 = row['jobtitle_1']
    jt2 = row['jobtitle_2']

    # Obtener opciones para cada columna
    opts1 = [x.strip() for x in jt1.split('/')]
    opts2 = [x.strip() for x in jt2.split('/')]

    # Crear combinaciones A × B
    for a, b in itertools.product(opts1, opts2):
        if a != b:
            expanded.append({
                't1': a,
                't2': b
            })

df_expanded = pd.DataFrame(expanded)

In [ ]:
# Normalización
df_expanded['t1'] = df_expanded['t1'].apply(normalize_text)
df_expanded['t2'] = df_expanded['t2'].apply(normalize_text)

# Eliminar duplicados (considerando pares desordenados)
# Creamos una columna temporal 'key' para filtrar
df_expanded['key'] = df_expanded.apply(lambda x: frozenset([x['t1'], x['t2']]), axis=1)
df_unique = df_expanded.drop_duplicates(subset='key').drop(columns=['key'])

# Filtrar vacíos
df_unique = df_unique[(df_unique['t1'] != "") & (df_unique['t2'] != "")]

# Convertir las columnas a string ANTES de crear el Dataset
df_unique['t1'] = df_unique['t1'].astype(str)
df_unique['t2'] = df_unique['t2'].astype(str)

# Resetear índice y crear el Dataset sin preservar el índice de pandas
df_unique = df_unique.reset_index(drop=True)
train_df_for_ds = df_unique[['t1', 't2']].rename(columns={'t1': 'sentence_1', 't2': 'sentence_2'})

In [ ]:
train_dataset = Dataset.from_pandas(train_df_for_ds, preserve_index=False)
print(f"Total de parejas en entrenamiento: {len(train_dataset)}")

Total de parejas en entrenamiento: 17855


### Validation data

In [ ]:
validation_dir = project_dir / 'data' / 'validation' / 'spanish'

corpus_val = pd.read_csv(validation_dir / 'corpus_elements', sep='\t')
queries_val = pd.read_csv(validation_dir / 'queries', sep='\t')
qrels_val = pd.read_csv(validation_dir / 'qrels.tsv', sep='\t', header=None)
qrels_val.columns = ['q_id', 'iter', 'doc_id', 'rel']

In [ ]:
# Diccionarios normalizados
corpus_dict = {str(row.iloc[0]): normalize_text(row.iloc[1]) for _, row in corpus_val.iterrows()}
queries_dict = {str(row.iloc[0]): normalize_text(row.iloc[1]) for _, row in queries_val.iterrows()}

# Construir dataset de pares positivos para calcular la Loss de validación
val_pairs = []
for _, row in qrels_val.iterrows():
    q_id = str(row.iloc[0])
    c_id = str(row.iloc[2])
    score = int(row.iloc[3])

    if score > 0 and q_id in queries_dict and c_id in corpus_dict:
        val_pairs.append({
            'sentence_1': queries_dict[q_id],
            'sentence_2': corpus_dict[c_id]
        })

In [ ]:
val_dataset = Dataset.from_list(val_pairs)
print(f"Pares de validación para Early Stopping: {len(val_dataset)}")

Pares de validación para Early Stopping: 7579


## Model preparation

In [ ]:
# Cargar modelo
model = SentenceTransformer(MODEL_NAME, device=DEVICE)

# Definir función de pérdida
# MultipleNegativesRankingLoss es estándar para pares (Anchor, Positive)
train_loss = losses.MultipleNegativesRankingLoss(model=model)

# Definir argumentos de entrenamiento (usando la clase nativa de SentenceTransformers/HF)
args = SentenceTransformerTrainingArguments(
    output_dir=str(model_output_path),
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    fp16=(DEVICE == 'cuda'),  # Usar precisión mixta si hay GPU
    eval_strategy="epoch",    # Evaluar al final de cada época
    save_strategy="epoch",    # Guardar checkpoint al final de cada época
    load_best_model_at_end=True, # Cargar el mejor modelo al terminar (CRUCIAL para Early Stopping)
    metric_for_best_model="eval_loss",
    save_total_limit=2,       # No llenar el disco con checkpoints
    logging_steps=50,
    batch_sampler=BatchSamplers.NO_DUPLICATES # Evitar duplicados en batch para MNRLoss
)

early_stopping_callback = EarlyStoppingCallback(
    early_stopping_patience=5,  # Número de épocas sin mejora antes de detener
    early_stopping_threshold=0.001 # Umbral mínimo de mejora
)

# Inicializar Trainer con EarlyStoppingCallback
trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    loss=train_loss,
    callbacks=[early_stopping_callback]
)

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

## Training model

In [ ]:
print("\n--- Comenzando entrenamiento... ---")
trainer.train()
print("\n--- Entrenamiento finalizado ---")


--- Comenzando entrenamiento... ---


Epoch,Training Loss,Validation Loss
1,0.471600,2.353265
2,0.224400,2.603937
3,0.157300,2.700533
4,0.128800,2.809389
5,0.070300,2.844799
6,0.061200,2.923309



--- Entrenamiento finalizado ---


In [ ]:
print(f"Guardando mejor modelo en: {best_model_path}")
model.save(str(best_model_path))

final_model = SentenceTransformer(str(best_model_path), device=DEVICE)

Guardando mejor modelo en: /content/output/2025-11-26/001/finetuned_models/paraphrase-multilingual-mpnet-base-v2/best_model


## Final evaluation

In [ ]:

# Ejecutar tu evaluación personalizada (si existe la función en el contexto)
# Asumimos que run_spanish_evaluation estaba definida o se importó.
# Si no, simplemente imprimimos confirmación.

print("\nModelo cargado y listo para inferencia.")
# Ejemplo de uso rápido:
# embeddings = final_model.encode(["ingeniero de datos", "data engineer"])
# sim = util.cos_sim(embeddings[0], embeddings[1])
# print(f"Similitud de prueba: {sim.item()}")


Modelo cargado y listo para inferencia.


In [ ]:
spanish_monolingual_evaluation(final_model, DEVICE, 'validation')

Loading Spanish data...
Encoding data: best_model on device: cuda


Batches:   0%|          | 0/6 [00:00<?, ?it/s]

Batches:   0%|          | 0/146 [00:00<?, ?it/s]

Calculating similarities and preparing results...
Evaluating Spanish monolingual performance...


/usr/local/lib/python3.12/dist-packages/ranx/metrics/average_precision.py:49: NumbaTypeSafetyWarning: unsafe cast from uint64 to int64. Precision may be lost.
  scores[i] = _average_precision(qrels[i], run[i], k, rel_lvl)


Spanish evaluation completed


{'map': np.float64(0.45862232062435015),
 'mrr': np.float64(0.5563063063063063),
 'ndcg': np.float64(0.745120995023976),
 'precision@5': np.float64(0.6605405405405405),
 'precision@10': np.float64(0.6545945945945946),
 'precision@100': np.float64(0.24421621621621625)}

In [ ]:
#comprimir carpeta del best_model
shutil.make_archive(best_model_path, 'zip', project_dir)

'/content/output/2025-11-26/001/finetuned_models/paraphrase-multilingual-mpnet-base-v2/best_model.zip'

In [ ]:
best_model_path

PosixPath('/content/output/2025-11-26/001/finetuned_models/paraphrase-multilingual-mpnet-base-v2/best_model')

# Massive Experimentation

In [ ]:
from huggingface_hub import list_models, model_info
from sentence_transformers import SentenceTransformer

device = 'cuda' if torch.cuda.is_available() else 'cpu'
source = 'validation'

models = list(list_models(
    pipeline_tag="sentence-similarity",
    library=["sentence-transformers", "transformers"],
    language="es"
))

models = sorted(models, key=lambda x: x.id)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_deprecation.py:100: FutureWarning: Deprecated argument(s) used in 'list_models': library, language. Will not be supported from version '1.0'.

Use `filter` instead.
  warnings.warn(message, FutureWarning)
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
df = pd.DataFrame(columns=['model_name', 'num_params', 'languages', 'map_es_es', 'mrr', 'ndcg', 'p@5', 'p@10', 'p@100'])
errors = []
for m in models:
  info = model_info(m.id)
  langs = info.cardData.get("languages") or info.cardData.get("language")
  mode = 'multilingual' if len(langs) > 1 else 'monolingual'

  try:
    model = SentenceTransformer(m.id, device=device, trust_remote_code=True)
    evaluation_results = spanish_monolingual_evaluation(model, device, source)
  except:
    errors.append(m.id)
    continue

  num_params = sum(p.numel() for p in model.parameters())

  df = pd.concat([df, pd.DataFrame([{
        'model_name': m.id,
        'num_params': num_params,
        'languages': mode,
        'map_es_es': evaluation_results.get('map', np.nan),
        'mrr': evaluation_results.get('mrr', np.nan),
        'ndcg': evaluation_results.get('ndcg', np.nan),
        'p@5': evaluation_results.get('precision@5', np.nan),
        'p@10': evaluation_results.get('precision@10', np.nan),
        'p@100': evaluation_results.get('precision@100', np.nan)
  }])], ignore_index=True)


df = df.sort_values('map_es_es', ascending=False).reset_index(drop=True)

Loading Spanish data...
Encoding data: pmmlv2-fine-tuned-flemish on device: cuda


Batches:   0%|          | 0/6 [00:00<?, ?it/s]

Batches:   0%|          | 0/146 [00:00<?, ?it/s]

Calculating similarities and preparing results...
Evaluating Spanish monolingual performance...
Spanish evaluation completed


/tmp/ipython-input-608551427.py:17: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, pd.DataFrame([{


...

config.json:   0%|          | 0.00/321 [00:00<?, ?B/s]

Loading Spanish data...
Encoding data: kochimetro-model on device: cuda


Batches:   0%|          | 0/6 [00:00<?, ?it/s]

Batches:   0%|          | 0/146 [00:00<?, ?it/s]

Calculating similarities and preparing results...
Evaluating Spanish monolingual performance...
Spanish evaluation completed


In [ ]:
df

,model_name,num_params,languages,map_es_es,mrr,ndcg,p@5,p@10,p@100
0,shtilev/medical_embedded_v4,278043648,multilingual,0.426456,0.545946,0.722666,0.647568,0.626486,0.230919
1,Francogv/Modelo-Embedding,278043648,multilingual,0.416908,0.551892,0.719501,0.640000,0.602162,0.226000
2,Dyna-99/local-st-model,278043648,multilingual,0.416908,0.551892,0.719501,0.640000,0.602162,0.226000
3,ammumadhu/kunato-indic-bert,278043648,multilingual,0.416908,0.551892,0.719501,0.640000,0.602162,0.226000
4,strauss-oak/mdl_bertopic_globo,278043648,multilingual,0.416908,0.551892,0.719501,0.640000,0.602162,0.226000
...,...,...,...,...,...,...,...,...,...
63,0xnu/pmmlv2-fine-tuned-igbo,117653760,multilingual,0.110013,0.486003,0.490691,0.368649,0.284324,0.076865
64,0xnu/pmmlv2-fine-tuned-hausa,117653760,multilingual,0.107842,0.469792,0.487138,0.371892,0.291892,0.075730
65,0xnu/pmmlv2-fine-tuned-yoruba,117653760,multilingual,0.105249,0.485193,0.485622,0.364324,0.277838,0.072865
66,0xnu/pmmlv2-fine-tuned-flemish,117653760,multilingual,0.099794,0.471253,0.481064,0.360000,0.264865,0.070757


In [ ]:
df.to_csv(output_dir / 'hf_evaluation_results.csv', index=False)

with open(output_dir / "hf_errors.txt", "w") as archivo:
    for item in errors:
        archivo.write(f"{item}\n")

## Errors management

In [ ]:
with open("hf_errors.txt", "r", encoding="utf-8") as f:
    lista = f.readlines()

lista = [line.strip() for line in lista]



In [ ]:
df = pd.DataFrame(columns=['model_name', 'num_params', 'languages', 'map_es_es', 'mrr', 'ndcg', 'p@5', 'p@10', 'p@100'])

errors = []
for model_name in lista:
  info = model_info(model_name)
  langs = info.cardData.get("languages") or info.cardData.get("language")
  mode = 'multilingual' if len(langs) > 1 else 'monolingual'

  try:
    model = SentenceTransformer(model_name, device=device, trust_remote_code=True)
    evaluation_results = spanish_monolingual_evaluation(model, device, source)
  except Exception as e:
    print(f"Error al cargar el modelo {model_name}")
    print(e)
    errors.append(model_name)
    continue

  num_params = sum(p.numel() for p in model.parameters())

  df = pd.concat([df, pd.DataFrame([{
        'model_name': model_name,
        'num_params': num_params,
        'languages': mode,
        'map_es_es': evaluation_results.get('map', np.nan),
        'mrr': evaluation_results.get('mrr', np.nan),
        'ndcg': evaluation_results.get('ndcg', np.nan),
        'p@5': evaluation_results.get('precision@5', np.nan),
        'p@10': evaluation_results.get('precision@10', np.nan),
        'p@100': evaluation_results.get('precision@100', np.nan)
  }])], ignore_index=True)


df = df.sort_values('map_es_es', ascending=False).reset_index(drop=True)

Some weights of the model checkpoint at Alibaba-NLP/gte-multilingual-base were not used when initializing NewModel: ['classifier.bias', 'classifier.weight']
- This IS expected if you are initializing NewModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing NewModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Loading Spanish data...
Encoding data: gte-multilingual-base on device: cuda


Batches:   0%|          | 0/6 [00:00<?, ?it/s]

Batches:   0%|          | 0/146 [00:00<?, ?it/s]

Calculating similarities and preparing results...
Evaluating Spanish monolingual performance...
Spanish evaluation completed


...

Error al cargar el modelo roytmana/paraphrase-multilingual-mpnet-base-v2-Q4_K_M-GGUF
roytmana/paraphrase-multilingual-mpnet-base-v2-Q4_K_M-GGUF does not appear to have a file named pytorch_model.bin, model.safetensors, tf_model.h5, model.ckpt or flax_model.msgpack.
Error al cargar el modelo sizrox/paraphrase-multilingual-mpnet-base-v2-Q8_0-GGUF
sizrox/paraphrase-multilingual-mpnet-base-v2-Q8_0-GGUF does not appear to have a file named pytorch_model.bin, model.safetensors, tf_model.h5, model.ckpt or flax_model.msgpack.


Error al cargar el modelo soprasteria/gte-multilingual-base
You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/soprasteria/gte-multilingual-base.
401 Client Error. (Request ID: Root=1-692ad354-4d2d1bfd5fc53d1d6a22213a;a497a90d-4743-43d6-ad58-9a586c60a726)

Cannot access gated repo for url https://huggingface.co/soprasteria/gte-multilingual-base/resolve/main/config.json.
Access to model soprasteria/gte-multilingual-base is restricted. You must have access to it and be authenticated to access it. Please log in.
Loading Spanish data...
Encoding data: gte-base-korean on device: cuda


Batches:   0%|          | 0/6 [00:00<?, ?it/s]

Batches:   0%|          | 0/146 [00:00<?, ?it/s]

Calculating similarities and preparing results...
Evaluating Spanish monolingual performance...
Spanish evaluation completed


In [ ]:
df.to_csv(output_dir / 'hf_evaluation_results_2.csv', index=False)

with open(output_dir / "hf_errors_2.txt", "w") as archivo:
    for item in errors:
        archivo.write(f"{item}\n")

In [ ]:
print(df)

                               model_name num_params     languages  map_es_es  \
0       Alibaba-NLP/gte-multilingual-base  305368320  multilingual   0.415536   
1  Jaume/gte-multilingual-base-no-network  305368320  multilingual   0.415536   
2                     leeloolee/intention  305368320  multilingual   0.415536   
3                  upskyy/gte-base-korean  305368320  multilingual   0.401787   

        mrr      ndcg       p@5      p@10     p@100  
0  0.541441  0.722481  0.627027  0.614595  0.226054  
1  0.541441  0.722481  0.627027  0.614595  0.226054  
2  0.541441  0.722481  0.627027  0.614595  0.226054  
3  0.563063  0.717950  0.627027  0.601622  0.221514  


In [ ]:
lista

['Alibaba-NLP/gte-multilingual-base',
 'EnverLee/bge-m3-korean-Q4_K_M-GGUF',
 'ImranzamanML/multilingual-mpnet-finetuned',
 'Jaeh1/bge-m3-korean-Q4_K_M-GGUF',
 'Jaume/gte-multilingual-base-no-network',
 'MagicalAlchemist/paraphrase-multilingual-mpnet-base-v2-Q8_0-GGUF',
 'Maxthemacaque/onnx-gte-multilingual-base',
 'Mignonbrothers/bge-m3-korean-Q4_K_M-GGUF',
 'Nagase-Kotono/e5-large-korean-Q8_0-GGUF',
 'armand01/paraphrase-multilingual-MiniLM-L12-v2-Q6_K-GGUF',
 'badger212/granite-embedding-278m-multilingual-Q4_K_M-GGUF',
 'coolsoon/e5-large-korean-Q4_K_M-GGUF',
 'cstr/paraphrase-multilingual-MiniLM-L12-v2-mlx',
 'fff1027/bge-m3-korean-Q4_K_M-GGUF',
 'hongkeon/bge-m3-korean-Q4_K_M-GGUF',
 'hongkeon/bge-m3-korean-Q8_0-GGUF',
 'hongkeon/e5-large-korean-Q8_0-GGUF',
 'leeloolee/intention',
 'mradermacher/ML-E5-0.3B-GGUF',
 'pocohos/paraphrase-multilingual-mpnet-base-v2-Q6_K-GGUF',
 'podarok/ukr-paraphrase-multilingual-mpnet-base',
 'pyarn/paraphrase-multilingual-mpnet-base-v2-Q8_0-GGUF',
 